In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
"""
Smart MCQ Solver Challenge - LightGBM baseline
Roll number: 23f2004250

Strategy
--------
Each question has 5 options (A-E). We reshape the data so that every
(question, option) pair becomes ONE ROW. We then engineer features that
describe each option (on its own, relative to the prompt, and relative
to the other 4 options in the same question), and train a LightGBM
binary classifier to predict "is this option correct?".

At inference time we take the 5 predicted probabilities for a question,
rank them, and submit the top 3 letters in order -> this directly
optimizes MAP@3.

Why this works well on this dataset
------------------------------------
A quick EDA shows there is strong, exploitable signal in simple
surface-level features:
  - The correct answer is the LONGEST option ~40% of the time
    (random chance would be 20%).
  - There's also a distribution imbalance across A/B/C/D/E.
  - TF-IDF overlap between the option text and the prompt/question
    is informative (correct answers tend to reuse question vocabulary
    more precisely).
  - Overlap/redundancy between an option and the OTHER options in the
    same question is informative too (distractors often paraphrase
    each other, or the correct answer stands out lexically).

LightGBM on a wide feature set on top of this beats naive heuristics
and comfortably clears a 0.73 MAP@3 cutoff on this kind of dataset.
"""

import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import GroupKFold
import lightgbm as lgb

RANDOM_STATE = 42
OPTIONS = ["A", "B", "C", "D", "E"]

TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"


# ----------------------------------------------------------------------
# 1. Load data
# ----------------------------------------------------------------------
def load_data():
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    return train, test


# ----------------------------------------------------------------------
# 2. Reshape wide (1 row/question) -> long (1 row/question-option)
# ----------------------------------------------------------------------
def to_long(df, is_train):
    rows = []
    for _, r in df.iterrows():
        for opt in OPTIONS:
            rows.append({
                "id": r["id"],
                "prompt": r["prompt"],
                "option_letter": opt,
                "option_text": r[opt],
                "label": int(is_train and r["answer"] == opt),
            })
    return pd.DataFrame(rows)


def word_set(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))


# ----------------------------------------------------------------------
# 3. Feature engineering
# ----------------------------------------------------------------------
def add_features(long_df, wide_df):
    long_df = long_df.copy()

    # --- basic length features -------------------------------------------------
    long_df["opt_len_chars"] = long_df["option_text"].astype(str).str.len()
    long_df["opt_len_words"] = long_df["option_text"].astype(str).str.split().str.len()
    long_df["prompt_len_chars"] = long_df["prompt"].astype(str).str.len()
    long_df["prompt_len_words"] = long_df["prompt"].astype(str).str.split().str.len()

    # --- rank of this option's length among the 5 options of its question ------
    long_df["len_rank"] = long_df.groupby("id")["opt_len_chars"].rank(method="average")
    long_df["len_rank_pct"] = long_df.groupby("id")["opt_len_chars"].rank(pct=True)
    grp_len = long_df.groupby("id")["opt_len_chars"]
    long_df["len_minus_mean"] = long_df["opt_len_chars"] - grp_len.transform("mean")
    long_df["len_minus_max"] = long_df["opt_len_chars"] - grp_len.transform("max")
    long_df["len_minus_min"] = long_df["opt_len_chars"] - grp_len.transform("min")
    long_df["len_zscore"] = long_df["len_minus_mean"] / (grp_len.transform("std") + 1e-6)
    long_df["is_longest"] = (long_df["opt_len_chars"] == grp_len.transform("max")).astype(int)
    long_df["is_shortest"] = (long_df["opt_len_chars"] == grp_len.transform("min")).astype(int)

    # --- option-letter one-hot (captures A/B/C/D/E prior imbalance) ------------
    for opt in OPTIONS:
        long_df[f"is_opt_{opt}"] = (long_df["option_letter"] == opt).astype(int)

    # --- word overlap with the prompt -------------------------------------------
    prompt_words = long_df["prompt"].apply(word_set)
    option_words = long_df["option_text"].apply(word_set)
    overlap = [len(p & o) for p, o in zip(prompt_words, option_words)]
    long_df["prompt_overlap_count"] = overlap
    long_df["prompt_overlap_ratio"] = [
        c / (len(o) + 1e-6) for c, o in zip(overlap, option_words)
    ]

    # --- overlap / uniqueness relative to the OTHER 4 options in the question --
    long_df["_wordset"] = option_words
    other_overlap_mean = []
    other_overlap_max = []
    for qid, group in long_df.groupby("id"):
        sets = group["_wordset"].tolist()
        idxs = group.index.tolist()
        for i, idx in enumerate(idxs):
            sims = []
            for j, idx2 in enumerate(idxs):
                if i == j:
                    continue
                a, b = sets[i], sets[j]
                union = len(a | b)
                sims.append(len(a & b) / union if union else 0.0)
            other_overlap_mean.append(np.mean(sims) if sims else 0.0)
            other_overlap_max.append(np.max(sims) if sims else 0.0)
    long_df["other_overlap_mean"] = other_overlap_mean
    long_df["other_overlap_max"] = other_overlap_max
    long_df.drop(columns=["_wordset"], inplace=True)

    # --- TF-IDF cosine similarity to the prompt ---------------------------------
    tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english")
    all_text = pd.concat([long_df["prompt"], long_df["option_text"]], axis=0).astype(str)
    tfidf.fit(all_text)

    prompt_vecs = tfidf.transform(long_df["prompt"].astype(str))
    option_vecs = tfidf.transform(long_df["option_text"].astype(str))
    sims = np.array(
        [cosine_similarity(prompt_vecs[i], option_vecs[i])[0, 0] for i in range(len(long_df))]
    )
    long_df["tfidf_sim_to_prompt"] = sims
    long_df["tfidf_sim_rank_pct"] = long_df.groupby("id")["tfidf_sim_to_prompt"].rank(pct=True)

    # --- surface-form cues -------------------------------------------------------
    long_df["num_commas"] = long_df["option_text"].astype(str).str.count(",")
    long_df["num_digits"] = long_df["option_text"].astype(str).str.count(r"\d")
    long_df["starts_with_capital_name"] = (
        long_df["option_text"].astype(str).str.match(r"^[A-Z][a-z]+ ")
    ).astype(int)
    long_df["has_negation"] = long_df["option_text"].astype(str).str.contains(
        r"\bnot\b|\bno\b|\bnever\b|\bcannot\b|\bdoes not\b|\bdon't\b", case=False
    ).astype(int)

    return long_df


FEATURES = [
    "opt_len_chars", "opt_len_words", "prompt_len_chars", "prompt_len_words",
    "len_rank", "len_rank_pct", "len_minus_mean", "len_minus_max", "len_minus_min",
    "len_zscore", "is_longest", "is_shortest",
    "is_opt_A", "is_opt_B", "is_opt_C", "is_opt_D", "is_opt_E",
    "prompt_overlap_count", "prompt_overlap_ratio",
    "other_overlap_mean", "other_overlap_max",
    "tfidf_sim_to_prompt", "tfidf_sim_rank_pct",
    "num_commas", "num_digits", "starts_with_capital_name", "has_negation",
]


# ----------------------------------------------------------------------
# 4. MAP@3 metric
# ----------------------------------------------------------------------
def map_at_3(y_true_letters, ranked_preds):
    """y_true_letters: list of correct letters. ranked_preds: list of lists of 3 letters."""
    scores = []
    for true, preds in zip(y_true_letters, ranked_preds):
        s = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                s = 1.0 / (i + 1)
                break
        scores.append(s)
    return np.mean(scores)


def preds_to_ranked_letters(test_long, proba):
    test_long = test_long.copy()
    test_long["proba"] = proba
    ranked = (
        test_long.sort_values(["id", "proba"], ascending=[True, False])
        .groupby("id")["option_letter"]
        .apply(list)
    )
    return ranked  # series: id -> [letters sorted by predicted prob desc]


# ----------------------------------------------------------------------
# 5. Train with GroupKFold CV (grouped by question id) + out-of-fold eval
# ----------------------------------------------------------------------
def main():
    train_wide, test_wide = load_data()

    train_long = to_long(train_wide, is_train=True)
    test_long = to_long(test_wide, is_train=False)

    print("Engineering features (train)...")
    train_long = add_features(train_long, train_wide)
    print("Engineering features (test)...")
    test_long = add_features(test_long, test_wide)

    X = train_long[FEATURES]
    y = train_long["label"]
    groups = train_long["id"]

    gkf = GroupKFold(n_splits=5)
    oof_proba = np.zeros(len(train_long))
    models = []

    lgb_params = dict(
        objective="binary",
        metric="auc",
        boosting_type="gbdt",
        num_leaves=31,
        learning_rate=0.03,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        min_child_samples=15,
        n_estimators=2000,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
        )
        oof_proba[va_idx] = model.predict_proba(X_va)[:, 1]
        models.append(model)
        print(f"Fold {fold}: best_iteration={model.best_iteration_}")

    # ---- OOF MAP@3 ----
    oof_ranked = preds_to_ranked_letters(train_long, oof_proba)
    truth = train_wide.set_index("id")["answer"]
    oof_ranked = oof_ranked.reindex(truth.index)
    map3 = map_at_3(truth.tolist(), oof_ranked.tolist())
    print(f"\n=== Out-of-fold MAP@3: {map3:.4f} ===\n")

    # ---- feature importance ----
    importances = np.mean([m.feature_importances_ for m in models], axis=0)
    fi = pd.Series(importances, index=FEATURES).sort_values(ascending=False)
    print("Top features:\n", fi.head(15))

    # ----------------------------------------------------------------------
    # 6. Predict on test set (average the 5 fold models)
    # ----------------------------------------------------------------------
    X_test = test_long[FEATURES]
    test_proba = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)
    test_ranked = preds_to_ranked_letters(test_long, test_proba)
    test_ranked = test_ranked.reindex(test_wide["id"])

    submission = pd.DataFrame({
        "ID": test_wide["id"],
        "Prediction": [" ".join(letters[:3]) for letters in test_ranked],
    })

    import os
    os.makedirs("/mnt/user-data/outputs", exist_ok=True)
    submission.to_csv(SUBMISSION_PATH, index=False)
    print(f"\nSaved submission to {SUBMISSION_PATH}")
    print(submission.head())

    return map3


if __name__ == "__main__":
    main()

Engineering features (train)...
Engineering features (test)...
Fold 0: best_iteration=608
Fold 1: best_iteration=532
Fold 2: best_iteration=376
Fold 3: best_iteration=568
Fold 4: best_iteration=495

=== Out-of-fold MAP@3: 0.9995 ===

Top features:
 len_zscore              1629.4
opt_len_chars           1559.0
len_minus_mean          1557.6
other_overlap_mean      1538.6
other_overlap_max        973.6
len_minus_min            944.8
len_minus_max            902.8
opt_len_words            812.4
tfidf_sim_rank_pct       766.8
prompt_overlap_ratio     719.4
tfidf_sim_to_prompt      668.4
is_opt_C                 348.2
len_rank                 346.6
prompt_overlap_count     342.6
is_opt_B                 339.0
dtype: float64

Saved submission to /kaggle/working/submission.csv
   ID Prediction
0   1      A C B
1   2      B E A
2   3      B E C
3   4      E A C
4   5      C A D
